# Ergodic BSDEs: What Happens as $T \to \infty$?

For long time horizons the value $Y_0$ of a BSDE grows linearly:

$$
Y_0(T) \approx \lambda \cdot T + v(X_0) + o(1) \quad \text{as } T\to\infty.
$$

The constant $\lambda$ is the **ergodic constant** — the long-run average of the
driver along the stationary distribution of the forward process.

An **Ergodic BSDE** directly characterises $(\lambda, v)$ without sending $T\to\infty$:

$$
Y_t = Y_T + \int_t^T [f(X_s, Y_s, Z_s) - \lambda]\,ds - \int_t^T Z_s\,dW_s
\quad \forall\,T > t.
$$

The Markovian ergodic PDE (nonlinear eigenvalue problem) is:

$$
\mathcal{L}v(x) + f(x, v(x), \sigma(x)v'(x)) = \lambda,
$$

where $\mathcal{L} = \tfrac{1}{2}\sigma^2\partial_{xx} + b\,\partial_x$.


In [ ]:
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

from ebsde.forward.ou_process import OrnsteinUhlenbeck
from ebsde.bsde.standard import StandardBSDE
from ebsde.bsde.ergodic import ErgodicBSDE
from ebsde.solvers.picard import PicardBSDESolver
from ebsde.solvers.ergodic_pde import ErgodicPDESolver
from data.synthetic import ergodic_ou_quadratic_analytical


## OU + Quadratic Driver: $f(x, y, z) = x^2 - \tfrac{1}{2}z^2$

This is the canonical benchmark.  Via the **Cole-Hopf transform**
$v = -2\log\phi$, the ergodic PDE linearises to a quantum harmonic
oscillator eigenvalue problem.  The exact ergodic constant is

$$
\lambda^* = \frac{1}{2}\left(-\kappa + \sqrt{\kappa^2 + 2\alpha\sigma^2}\right)
\approx 0.366 \quad (\kappa=\sigma=\alpha=\gamma=1).
$$


In [ ]:
import os
# --- Setup ----------------------------------------------------------
ou = OrnsteinUhlenbeck(kappa=1.0, theta=0.0, sigma=1.0)
driver_q = lambda x, y, z: x**2 - 0.5 * np.sum(np.asarray(z, dtype=float)**2, axis=-1)
terminal  = lambda x: np.zeros_like(np.asarray(x, dtype=float))

# --- Exact ergodic constant via Cole-Hopf --------------------------
exact = ergodic_ou_quadratic_analytical(
    kappa=1.0, sigma=1.0, alpha=1.0, beta=0.0, gamma=1.0
)
lambda_exact = exact['lambda_ergodic']
print(f"Exact lambda* = {lambda_exact:.6f}")

# --- Y0(T) grows linearly with T -----------------------------------
T_values = [5, 10, 20, 40]
Y0_list  = []
for T in T_values:
    bsde_T = StandardBSDE(forward=ou, driver=driver_q, terminal=terminal, T=float(T))
    sol    = PicardBSDESolver(bsde_T, n_paths=10000, n_steps=50, n_picard=3).solve()
    Y0_list.append(float(sol['Y0']))
    print(f"  T={T:>4}  Y0={sol['Y0']:.4f}  Y0/T={sol['Y0']/T:.4f}")

# --- Plot Y0 vs T with linear fit ----------------------------------
T_arr  = np.array(T_values, dtype=float)
Y0_arr = np.array(Y0_list)
slope  = np.polyfit(T_arr, Y0_arr, 1)[0]
print(f"\nLinear slope (Y0/T) = {slope:.6f}  (exact lambda* = {lambda_exact:.6f})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(T_arr, Y0_arr, 'o-', lw=2, ms=8, label='Y0(T)')
ax.plot(T_arr, slope * T_arr, '--', color='gray', label=f'slope = {slope:.3f}')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('T')
ax.set_ylabel('Y_0(T)')
ax.set_title('Y_0(T) grows linearly — slope → λ*')
ax.legend()
fig.tight_layout()

os.makedirs('notebooks/figures', exist_ok=True)
fig.savefig('notebooks/figures/02_y0_vs_T.png', dpi=120)
plt.close(fig)
print("Figure saved → notebooks/figures/02_y0_vs_T.png")


## The Ergodic Constant via `ErgodicPDESolver`

Instead of computing $Y_0(T)$ for large $T$, the `ErgodicPDESolver` directly
solves the nonlinear eigenvalue problem to find $\lambda$ and the potential
function $v(x)$ (normalised so that $v(0)=0$).


In [ ]:
# --- Solve ergodic PDE ---------------------------------------------
ebsde = ErgodicBSDE(forward=ou, driver=driver_q)
epde  = ErgodicPDESolver(ebsde, n_x=300).solve()

lambda_pde = epde['lambda_ergodic']
v_grid     = epde['v']
x_grid     = epde['x_grid']

print(f"ErgodicPDESolver lambda = {lambda_pde:.6f}")

# --- Plot v(x) ------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(x_grid, v_grid, 'C1', lw=2)
ax.set_xlabel('x')
ax.set_ylabel('v(x)')
ax.set_title(f'Ergodic value function  (λ = {lambda_pde:.4f})')
ax.axhline(0, color='k', lw=0.5, ls='--')
fig.tight_layout()
fig.savefig('notebooks/figures/02_value_function.png', dpi=120)
plt.close(fig)
print("Figure saved → notebooks/figures/02_value_function.png")


In [ ]:
print("=" * 45)
print(f"{'Method':<30} {'lambda':>10}")
print("-" * 45)
print(f"{'Exact (Cole-Hopf)':<30} {lambda_exact:>10.6f}")
print(f"{'ErgodicPDESolver':<30} {lambda_pde:>10.6f}")
print(f"{'|Error|':<30} {abs(lambda_pde - lambda_exact):>10.2e}")
print("=" * 45)
